In [ ]:
%pip install -q textstat pandas

In [ ]:
import re


def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        u"\U0001FA00-\U0001FA6F"
        u"\U00002500-\U00002BFF"
        u"\U00002600-\U000026FF"
        u"✀-➿"
        u"⌀-⏿"
        "✓✗⚠►▸▼▲◄→←↑↓│├└┌┐┘┤┬┴┼"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)


def replace_arrows(text):
    if not isinstance(text, str):
        return ''
    arrows = [
        ('-->', ' leads to '),
        ('=>',  ' leads to '),
        ('→',   ' leads to '),
        ('►',   ' leads to '),
        ('▼',   ' then '),
        ('▲',   ' then '),
        ('←',   ' comes from '),
    ]
    for arrow, replacement in arrows:
        text = text.replace(arrow, replacement)
    return text


def extract_code_block_content(text):
    def clean_block(match):
        code = match.group(1)
        code = ''.join(
            ' ' if (0x2500 <= ord(ch) <= 0x257F) else ch
            for ch in code
        )
        code = re.sub(r'[ \t]+', ' ', code)
        code = re.sub(r'\n{2,}', '\n', code)
        return code.strip()

    return re.sub(
        r'```(?:\w+)?\n?(.*?)```',
        clean_block,
        text,
        flags=re.DOTALL
    )


def convert_table_to_text_safe(text):
    lines = text.split('\n')
    result = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('|') and line.endswith('|'):
            if re.match(r'^\|[\s\-:]+\|$', line):
                i += 1
                continue
            cells = [c.strip() for c in line.strip('|').split('|') if c.strip()]
            if i + 1 < len(lines) and re.match(r'^\|[\s\-:|]+\|$', lines[i + 1].strip()):
                headers = cells
                i += 2
                while i < len(lines):
                    row = lines[i].strip()
                    if not (row.startswith('|') and row.endswith('|')):
                        break
                    values = [c.strip() for c in row.strip('|').split('|') if c.strip()]
                    headers = [re.sub(r'\*+(.+?)\*+', r'\1', h) for h in headers]
                    values  = [re.sub(r'\*+(.+?)\*+', r'\1', v) for v in values]
                    pairs = []
                    for h, v in zip(headers, values):
                        if h and v:
                            pairs.append(f"{h} is {v}")
                    if pairs:
                        result.append('. '.join(pairs) + '.')
                    i += 1
            else:
                result.append('. '.join(cells) + '.')
                i += 1
        else:
            result.append(lines[i])
            i += 1
    return '\n'.join(result)


def remove_markdown_dividers(text):
    return re.sub(r'^\s*[-_*]{2,}\s*$', '', text, flags=re.MULTILINE)


def ensure_line_terminators(text):
    """Newline-as-sentence rule (FK only).
    Each non-empty line is treated as one sentence. If it does not already
    end in [.!?:], append '.'. No length filter, no skipping. Applied
    identically to every model so format style is measured, not engineered.
    """
    lines = text.split('\n')
    out = []
    for line in lines:
        s = line.strip()
        if not s:
            out.append('')
            continue
        m = re.match(r'^(\-|\*|•|\d+[\.\)])\s+(.+)', s)
        if m:
            s = m.group(2).strip()
        if s and s[-1] not in '.!?:':
            s += '.'
        out.append(s)
    return '\n'.join(out)


def clean_markdown(text):
    if not isinstance(text, str):
        return ''

    text = replace_arrows(text)
    text = remove_emoji(text)

    text = extract_code_block_content(text)

    text = convert_table_to_text_safe(text)

    text = remove_markdown_dividers(text)

    text = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
    text = re.sub(r'^[\-\*\+]\s+', '', text, flags=re.MULTILINE)

    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")
    text = re.sub(r'"{2,}', '"', text)
    text = text.replace('"', '')
    text = text.replace('—', ' ')
    text = text.replace('–', ' ')

   
    text = re.sub(r'#+\s*', '', text)

    text = ensure_line_terminators(text)

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [ ]:
import re
import os
import pandas as pd
import textstat

In [ ]:
INPUT_DIR  = "../Dataset/responses"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Working dir:", os.getcwd())
print("Data dir   :", INPUT_DIR)
print("Output dir :", OUTPUT_DIR)

In [ ]:
def run_fk(input_path, output_path, label=None):
    df = pd.read_csv(input_path, sep='\t')
    df['LLM_Response'] = df['LLM_Response'].astype(str)

    model_col = 'Model_Name' if 'Model_Name' in df.columns else 'Answer_Model'
    if label is None:
        label = os.path.basename(input_path)

    print(f'\n[{label}] Loaded {len(df)} rows')
    print(f'[{label}] Models: {df[model_col].unique().tolist()}')
    print(f'[{label}] Levels: {sorted(df["Literacy_Level"].unique().tolist())}')

    df['Clean_Response'] = df['LLM_Response'].apply(clean_markdown)
    df['Word_Count']     = df['Clean_Response'].apply(lambda x: len(x.split()))
    df['FK_Grade']       = df['Clean_Response'].apply(textstat.flesch_kincaid_grade)

    keep = ['ID', 'Domain', 'Topic', model_col, 'Literacy_Level', 'Word_Count', 'FK_Grade']
    if 'Question_Model' in df.columns:
        keep.insert(3, 'Question_Model')
    df[keep].to_csv(output_path, sep='\t', index=False)
    print(f'[{label}] Saved -> {output_path}')

    summary = df.groupby([model_col, 'Literacy_Level']).agg(
        Avg_FK_Grade   = ('FK_Grade',   'mean'),
        Avg_Word_Count = ('Word_Count', 'mean'),
        Count          = ('FK_Grade',   'count'),
    ).round(2)
    print(f'\n[{label}] AVG FK GRADE & WORD COUNT')
    print(summary.to_string())

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "explicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_claude_explicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "explicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_gemini_explicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "explicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_gpt_explicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "explicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_llama_explicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "explicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_qwen_explicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "implicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_claude_implicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "implicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_gemini_implicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "implicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_gpt_implicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "implicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_llama_implicit.tsv"),
)

In [ ]:
run_fk(
    input_path  = os.path.join(INPUT_DIR,  "implicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "Readability_qwen_implicit.tsv"),
)